In [5]:
from torch.nn.functional import softmax
import torch
import torch.nn as nn
import os
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
path = os.getcwd()

In [6]:
GPT2Config = {
  "activation_function": "gelu_new", # 'new'?
  "architectures": [
    "GPT2LMHeadModel" # anything special bout this?
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256, # same as eos?
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "resid_pdrop": 0.1,
  "summary_activation": None,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": True,
  "summary_type": "cls_index",
  "summary_use_proj": True,
  "task_specific_params": {
    "text-generation": {
      "do_sample": True,
      "max_length": 50
    }
  },
  "vocab_size": 50257
}

In [7]:
state_dict = torch.load(path+'/weights/pytorch_model.bin')

In [221]:
def scaled_dot_product_attention(queries, keys, values, mask=None):
    """
    - Q, K, V each (n_tokens x out_dim)
    - Q @ K.T => (n_tokens x n_tokens) (similarity score)
    - (Q @ K.T) @ V => (QK^T: n_tokens x n_tokens) x (V: n_tokens x out_dim)
                    => (n_tokens x out_dim) returned (attn)
    - DIM NOT CHANGED by SDP
    """
    similarity_score = queries.matmul(keys.T)
    dk = keys.size(-1)
    denom = torch.sqrt(torch.tensor(dk))
    sdp = 1/denom * similarity_score
    
    if mask is not None:
        n_tokens = keys.size(-2) # <= n_ctx
        mask_val = torch.finfo(sdp.dtype).min # something like -3.4e38 for float32
        masked_attn_weights = torch.where(mask[:n_tokens, :n_tokens], sdp, mask_val)
        sdp = masked_attn_weights

    attn = torch.matmul(softmax(sdp, dim=-1), values)
    return attn

class AttentionHead(nn.Module):
    def __init__(self, in_dim, out_dim): # both equal to embedding_dim- no, out_dim=head_dim
        super().__init__()
        """
        - DIM NOT CHANGED by these
        - in: n_tokens x embed_dim
        - out: n_tokens x embed_dim
        TODO: batch index
        """
        self.Q_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        self.K_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        self.V_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        
    def forward(self, x, mask=None): # n_tokens x embed_dim
        Q = self.Q_proj(x) # (x: n_tokens x embed_dim) @ (Q_proj: embed_dim x embed_dim)^T -> (n_tokens x embed_dim)
        K = self.K_proj(x) # dim same as x (")
        V = self.V_proj(x) # dim same as x (")
        return scaled_dot_product_attention(Q, K, V, mask=mask)

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, in_dim, out_dim, n_ctx):
        super().__init__()
        self.head_dim = out_dim // n_heads # 768/12 = 64 head_dim
        self.attn_heads = nn.ModuleList([
            # each head takes in (n_tokens x head_dim)
            # and returns        (n_tokens x head_dim)
            AttentionHead(in_dim, self.head_dim) for _ in range(n_heads)
        ])
        self.register_buffer(
            "mask",
            torch.tril(torch.ones((n_ctx, n_ctx), dtype=torch.bool)),
            persistent=False
        )

        self.out_proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        n_tokens, in_dim = x.shape
        heads = torch.concat([head(x, mask=self.mask) for head in self.attn_heads], dim=-1)
        # = n_tokens x (head_dim*n_heads = out_dim = embed_dim)
        out = self.out_proj(heads)
        return out

class TransformerBlock(nn.Module):
    
    def __init__(self, n_heads, in_dim, out_dim, n_ctx):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, in_dim, out_dim, n_ctx) # returns in_dim x dk*n_heads
        self.layernorm1 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.layernorm2 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.dropout = nn.Dropout(GPT2Config['attn_pdrop'])
        
        self.ffn = nn.Sequential(
            nn.Linear(in_dim, in_dim * 4),
            nn.GELU(approximate='tanh'),
            nn.Linear(in_dim * 4, in_dim)
        )

    def forward(self, x):
        x_old = x
        x = self.layernorm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + x_old
        
        x_old_2 = x
        x = self.layernorm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = x + x_old_2
        return x

class TransformerDecoder(nn.Module):
    
    def __init__(self, n_layers, n_heads, vocab_size, n_embed, n_ctx):
        super().__init__()
        self.ctx_len = n_ctx
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding = nn.Embedding(n_ctx, n_embed)
        self.dropout = nn.Dropout(0.1)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_heads, n_embed, n_embed, n_ctx) 
            for _ in range(n_layers)
        ])
        self.layernorm_final = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.linear = nn.Linear(n_embed, vocab_size, bias=False) # transpose of token_embedding; weights tied
        
        self.softmax = nn.Softmax(dim=0)
        
    def forward(self, x):
        seq_len = x.shape[-1]
        assert seq_len <= self.ctx_len
        positions = torch.arange(seq_len) # 0, 1, 2, .., seq_len-1
        
        embeddings = self.token_embedding(x) + self.positional_embedding(positions)
        
        x = self.dropout(embeddings)
        x = self.blocks(x)
        x = self.layernorm_final(x)
        x = self.linear(x)
        
        return x

In [222]:
def generate_text_simple(model, idx, max_new_tokens, context_size, temp=1.0, k=40):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[-context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[-1, :] / temp

        # Apply softmax to get probabilities
        #probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)
        #print(probas.max())

        # Get the idx of the vocab entry with the highest probability value
        #idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)
        #print(logits.shape)
        topk_logits, topk_indices = torch.topk(logits, k)
        topk_probs = torch.softmax(topk_logits, dim=-1)
        sampled_index = torch.multinomial(topk_probs, num_samples=1)
        idx_next = topk_indices[sampled_index]

        #idx_next = torch.multinomial(logits, 1, dim=-1)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=0)  # (batch, n_tokens+1)
    return idx

In [116]:
from typing import OrderedDict

def _load_weights(transformer: TransformerDecoder, 
                  state_dict: OrderedDict,
                  n_layers: int,
                  n_heads: int):
    # load embedding matrices
    transformer.load_state_dict({
        f'token_embedding.weight': state_dict[f'wte.weight'],
        f'positional_embedding.weight': state_dict[f'wpe.weight'],
        f'linear.weight': state_dict[f'wte.weight'], # tied/shared with input embedding weights; see markdown below
    }, strict=False)

    # load last layernorm
    transformer.load_state_dict({
        f'layernorm_final.weight': state_dict[f'ln_f.weight'],
        f'layernorm_final.bias': state_dict[f'ln_f.bias'],
    }, strict=False)

    # for each block,
    # load the two layernorms, 
    # the projection weights 
    # and the attention QKV params for each head
    # (skip the mask in state_dict['h.0.attn.bias'] since it's the same..)
    
    for block_idx in range(n_layers):
        transformer.load_state_dict({
            f'blocks.{block_idx}.layernorm1.weight': state_dict[f'h.{block_idx}.ln_1.weight'],
            f'blocks.{block_idx}.layernorm1.bias': state_dict[f'h.{block_idx}.ln_1.bias'],
            f'blocks.{block_idx}.layernorm2.weight': state_dict[f'h.{block_idx}.ln_2.weight'],
            f'blocks.{block_idx}.layernorm2.bias': state_dict[f'h.{block_idx}.ln_2.bias'],
            f'blocks.{block_idx}.mha.out_proj.weight': state_dict[f'h.{block_idx}.attn.c_proj.weight'].T, # transpose cuz Conv1D
            f'blocks.{block_idx}.mha.out_proj.bias': state_dict[f'h.{block_idx}.attn.c_proj.bias'],
            f'blocks.{block_idx}.ffn.0.weight': state_dict[f'h.{block_idx}.mlp.c_fc.weight'].T, # transposed cuz Conv1D
            f'blocks.{block_idx}.ffn.0.bias': state_dict[f'h.{block_idx}.mlp.c_fc.bias'],
            f'blocks.{block_idx}.ffn.2.weight': state_dict[f'h.{block_idx}.mlp.c_proj.weight'].T, # transposed cuz Conv1D
            f'blocks.{block_idx}.ffn.2.bias': state_dict[f'h.{block_idx}.mlp.c_proj.bias'],
        }, strict=False)
            
        # split from 2304 to three with dim 768 (64 * 12 heads in each matrix)
        Q_proj_w, K_proj_w, V_proj_w = state_dict[f'h.{block_idx}.attn.c_attn.weight'].split(768, dim=1)
        Q_proj_b, K_proj_b, V_proj_b = state_dict[f'h.{block_idx}.attn.c_attn.bias'].split(768, dim=0)
        
        # for each head, split its weights from the full matrix, then load into MHA state_dict
        for head_idx in range(n_heads):
            transformer.load_state_dict({
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.Q_proj.weight': Q_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.K_proj.weight': K_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.V_proj.weight': V_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.Q_proj.bias': Q_proj_b.split(64, dim=0)[head_idx],
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.K_proj.bias': K_proj_b.split(64, dim=0)[head_idx],
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.V_proj.bias': V_proj_b.split(64, dim=0)[head_idx]
            }, strict=False)

In [223]:
transformer = TransformerDecoder(
    n_layers=12, 
    n_heads=12,
    vocab_size=50257, 
    n_embed=768,
    n_ctx=1024
)
transformer.state_dict()['blocks.0.mha.attn_heads.0.Q_proj.weight']

tensor([[ 0.0059, -0.0091,  0.0097,  ...,  0.0353, -0.0262, -0.0114],
        [-0.0110,  0.0194,  0.0329,  ...,  0.0069,  0.0354,  0.0035],
        [-0.0313,  0.0105, -0.0281,  ..., -0.0051, -0.0022,  0.0016],
        ...,
        [-0.0270, -0.0156, -0.0275,  ...,  0.0096, -0.0180,  0.0214],
        [ 0.0303,  0.0010, -0.0153,  ..., -0.0174, -0.0132, -0.0152],
        [ 0.0068, -0.0145,  0.0348,  ...,  0.0033, -0.0227, -0.0104]])

In [224]:
_load_weights(transformer, state_dict, n_layers=12, n_heads=12)

In [254]:
torch.random.manual_seed(42)
token_input = torch.tensor(tokenizer.encode("Hello, I'm an AI!"))
transformer.eval()
res = generate_text_simple(transformer, token_input, max_new_tokens=40, context_size=1024, k=40, temp=0.8)
print("Generated text:\n\n", tokenizer.decode(res.tolist()))

Generated text:

 Hello, I'm an AI! I've made myself a great friend - but a real human being! - to you.

There is a great deal of pressure to be a human. The world around you is full of pain


In [104]:
transformer.state_dict()['blocks.0.mha.attn_heads.0.Q_proj.weight']

tensor([[-0.4738, -0.2614, -0.0978,  ...,  0.3237, -0.0483, -0.2235],
        [ 0.0874,  0.1473,  0.2387,  ..., -0.0770, -0.1492,  0.1507],
        [ 0.0039,  0.0695,  0.3668,  ..., -0.1235, -0.1660, -0.0480],
        ...,
        [ 0.0994,  0.2158,  0.1541,  ..., -0.1665, -0.0951,  0.1028],
        [-0.3707,  0.0330, -0.1596,  ...,  0.3551, -0.0443,  0.2452],
        [-0.2440,  0.1519,  0.3106,  ...,  0.1196, -0.0738, -0.1380]])

In [108]:
state_dict['h.0.attn.c_attn.weight']

tensor([[-0.4738, -0.2614, -0.0978,  ...,  0.0513, -0.0584,  0.0250],
        [ 0.0874,  0.1473,  0.2387,  ..., -0.0525, -0.0113, -0.0156],
        [ 0.0039,  0.0695,  0.3668,  ...,  0.1143,  0.0363, -0.0318],
        ...,
        [-0.2592, -0.0164,  0.1991,  ...,  0.0095, -0.0516,  0.0319],
        [ 0.1517,  0.2170,  0.1043,  ...,  0.0293, -0.0429, -0.0475],
        [-0.4100, -0.1924, -0.2400,  ..., -0.0046,  0.0070,  0.0198]])

In [106]:
 state_dict[f'h.0.attn.c_attn.weight'].shape

torch.Size([768, 2304])

In [148]:
token_input = torch.tensor(tokenizer.encode("Hello, I'm a language model,"))
transformer.eval()
res = generate_text_simple(transformer, token_input, max_new_tokens=6, context_size=1024)
print("Generated text:\n\n", tokenizer.decode(res.tolist()))

Generated text:

 Thank! ThankYOURSELF!!!!


In [38]:
state_dict['h.0.mlp.c_fc.weight'].shape, state_dict['h.0.mlp.c_proj.bias'].shape

(torch.Size([768, 3072]), torch.Size([3072]))

In [80]:
transformer.blocks[0].ffn[0].weight.shape

torch.Size([3072, 768])

In [66]:
transformer

TransformerDecoder(
  (token_embedding): Embedding(50257, 768)
  (positional_embedding): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): Sequential(
    (0): TransformerBlock(
      (mha): MultiHeadAttention(
        (attn_heads): ModuleList(
          (0-11): 12 x AttentionHead(
            (Q_proj): Linear(in_features=768, out_features=64, bias=True)
            (K_proj): Linear(in_features=768, out_features=64, bias=True)
            (V_proj): Linear(in_features=768, out_features=64, bias=True)
          )
        )
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (layernorm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layernorm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (ffn): Sequential(
        (0): Linear(in_features=768, out_features=3072, bias=True)
        (1): GELU(approximate='tanh')
        (2): Linear(in_features=3072, out

In [220]:
torch.tensor([0.9, 0.1]).topk(k=2)

torch.return_types.topk(
values=tensor([0.9000, 0.1000]),
indices=tensor([0, 1]))